# AE3 – Transformación y enriquecimiento de datos
**Proyecto:** Caracterización descriptiva de los precios de arriendo de apartamentos por ciudad y barrio en Colombia  
**Estudiante:** Duván Oswaldo Ramírez Duarte  
**Dataset:** `aptologia/precios-arriendo-colombia`  
**Configuración:** `precios_arriendo_barrios` · **Split:** `train`

Este notebook continúa la AE2. Conserva una copia original sin modificar (`df_original`), transforma una copia de trabajo, incorpora indicadores derivados, responde preguntas descriptivas con tablas y exporta los archivos de AE3.

**Unidad de análisis:** registro agregado por ciudad y barrio en la fotografía disponible. Los precios son precios pedidos publicados en avisos, no precios finales de contratos.


## 1. Necesidad, preguntas de negocio y variables

**Necesidad retomada de AE2:** disponer de referencias descriptivas comparables sobre la mediana mensual del precio pedido de arriendo y su dispersión central por ciudad y barrio, dentro de la cobertura del dataset.

**Pregunta principal de AE2:** ¿Cómo varía la mediana mensual del precio pedido de arriendo de apartamentos entre las ciudades y los barrios incluidos en el dataset, y qué diferencias descriptivas se observan entre los percentiles 25 y 75?

Para responderla, se operacionaliza en tres preguntas de trabajo (no cambia la necesidad; divide la pregunta original en resultados verificables):

1. **Comparación por ciudad:** ¿Cómo se distribuyen las medianas de precio de los barrios incluidos en cada ciudad y cuántos avisos respaldan esos registros?
2. **Comparación por barrio:** ¿Qué barrios presentan las medianas de precio pedido más altas y más bajas dentro de los registros disponibles?
3. **Dispersión central:** ¿Qué amplitud tiene el intervalo entre p25 y p75 por barrio y cómo varía entre ciudades?

Variables fuente: `fecha`, `ciudad_id`, `ciudad`, `barrio`, `avisos`, `p25`, `mediana`, `p75`, `con_parqueadero`, `con_telefono`.  
Indicadores derivados: `rango_intercuartil`, `rango_intercuartil_pct_mediana`, `participacion_avisos_parqueadero`, `participacion_avisos_telefono`, y campos normalizados de ciudad/barrio.


## 2. Preparar el entorno
Se fija pandas en una versión compatible con Google Colab.

In [1]:
%pip -q install -U datasets "pandas==2.2.3"


In [2]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from datasets import load_dataset
from IPython.display import display

DATASET_ID = "aptologia/precios-arriendo-colombia"
CONFIGURACION = "precios_arriendo_barrios"
SPLIT = "train"
CARPETA = Path("AE3")
CARPETA_DATOS = CARPETA / "datos"
CARPETA_RESULTADOS = CARPETA / "resultados"
CARPETA_DATOS.mkdir(parents=True, exist_ok=True)
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)


## 3. Cargar el dataset y conservar el original

In [3]:
dataset_hf = load_dataset(DATASET_ID, CONFIGURACION, split=SPLIT)
df_original = dataset_hf.to_pandas().copy(deep=True)
if df_original.empty:
    raise ValueError("El dataset cargado está vacío.")
df = df_original.copy(deep=True)

print("Dimensiones originales:", df_original.shape)
display(df_original.head())


README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

precios_arriendo_barrios.csv:   0%|          | 0.00/110k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1460 [00:00<?, ? examples/s]

Dimensiones originales: (1460, 10)


,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841
2,2026-10-05,medellin,Medellín,Laureles,5780,3300000,4200000,5300000,4283,2169
3,2026-10-05,medellin,Medellín,Castropol,2730,4200000,5350000,7000000,2286,1547
4,2026-10-05,medellin,Medellín,El Tesoro,2471,5800000,7700000,11500000,2167,1278


## 4. Diagnóstico inicial de calidad
Se registran filas, columnas, tipos, nulos, duplicados exactos y duplicados en la clave esperada. No se eliminan registros automáticamente: cualquier tratamiento se condiciona a la evidencia.


In [4]:
diagnostico_antes = {
    "filas": len(df),
    "columnas": len(df.columns),
    "nulos": int(df.isna().sum().sum()),
    "duplicados_exactos": int(df.duplicated().sum())
}
print(diagnostico_antes)
display(df.dtypes.rename("tipo_antes").to_frame())
display(df.isna().sum().rename("nulos_antes").to_frame())

clave = [c for c in ["fecha", "ciudad_id", "barrio"] if c in df.columns]
duplicados_clave_antes = int(df.duplicated(subset=clave).sum()) if len(clave) == 3 else None
print("Duplicados en clave fecha-ciudad_id-barrio:", duplicados_clave_antes)


{'filas': 1460, 'columnas': 10, 'nulos': 0, 'duplicados_exactos': 0}


,tipo_antes
fecha,object
ciudad_id,object
ciudad,object
barrio,object
avisos,int64
p25,int64
mediana,int64
p75,int64
con_parqueadero,int64
con_telefono,int64


,nulos_antes
fecha,0
ciudad_id,0
ciudad,0
barrio,0
avisos,0
p25,0
mediana,0
p75,0
con_parqueadero,0
con_telefono,0


Duplicados en clave fecha-ciudad_id-barrio: 0


## 5. Transformación y validación
Reglas:
- `fecha` se convierte a fecha; los identificadores y nombres se conservan como texto y se normalizan en columnas auxiliares sin sobrescribir los valores fuente.
- Las variables de conteo y precio se convierten a numéricas.
- Se comprueba la coherencia `p25 <= mediana <= p75`, conteos no negativos y conteos de atributos no superiores a `avisos`.
- Se informan faltantes, duplicados y registros inconsistentes. No se imputan nulos ni se eliminan filas de forma silenciosa.


In [5]:
columnas_numericas = ["avisos", "p25", "mediana", "p75", "con_parqueadero", "con_telefono"]
for col in columnas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "fecha" in df.columns:
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

for col in ["ciudad", "barrio", "ciudad_id"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

df["ciudad_normalizada"] = df["ciudad"].str.normalize("NFKC").str.casefold().str.strip()
df["barrio_normalizado"] = df["barrio"].str.normalize("NFKC").str.casefold().str.strip()

faltantes_despues_conversion = df.isna().sum()
duplicados_exactos_despues = int(df.duplicated().sum())
duplicados_clave_despues = int(df.duplicated(subset=clave).sum()) if len(clave) == 3 else None

orden_percentiles_valido = (
    df[["p25", "mediana", "p75"]].notna().all(axis=1)
    & (df["p25"] <= df["mediana"])
    & (df["mediana"] <= df["p75"])
)
conteos_no_negativos = (df[["avisos", "con_parqueadero", "con_telefono"]].fillna(0) >= 0).all(axis=1)
atributos_no_superan_avisos = (
    (df["con_parqueadero"] <= df["avisos"])
    & (df["con_telefono"] <= df["avisos"])
)
registros_inconsistentes = ~(orden_percentiles_valido & conteos_no_negativos & atributos_no_superan_avisos)

print("Filas después de conversión (no se eliminan filas):", len(df))
print("Nulos después de conversión:")
display(faltantes_despues_conversion.rename("nulos").to_frame())
print("Duplicados exactos:", duplicados_exactos_despues)
print("Duplicados de clave:", duplicados_clave_despues)
print("Registros con alguna regla de consistencia incumplida:", int(registros_inconsistentes.sum()))


Filas después de conversión (no se eliminan filas): 1460
Nulos después de conversión:


,nulos
fecha,0
ciudad_id,0
ciudad,0
barrio,0
avisos,0
p25,0
mediana,0
p75,0
con_parqueadero,0
con_telefono,0


Duplicados exactos: 0
Duplicados de clave: 0
Registros con alguna regla de consistencia incumplida: 0


## 6. Enriquecimiento: indicadores derivados
El rango intercuartílico se calcula como `p75 - p25` (COP/mes). Su versión relativa divide ese rango por la mediana y se expresa como porcentaje. Las participaciones de avisos con parqueadero y teléfono se calculan sobre `avisos`; solo se calculan cuando el denominador es positivo. Estos indicadores describen los conteos reportados y no estiman efectos causales sobre el precio.


In [6]:
df["rango_intercuartil"] = df["p75"] - df["p25"]
df["rango_intercuartil_pct_mediana"] = np.where(
    df["mediana"] > 0,
    df["rango_intercuartil"] / df["mediana"] * 100,
    np.nan
)
df["participacion_avisos_parqueadero"] = np.where(
    df["avisos"] > 0,
    df["con_parqueadero"] / df["avisos"] * 100,
    np.nan
)
df["participacion_avisos_telefono"] = np.where(
    df["avisos"] > 0,
    df["con_telefono"] / df["avisos"] * 100,
    np.nan
)
display(df[["ciudad", "barrio", "p25", "mediana", "p75", "rango_intercuartil",
            "rango_intercuartil_pct_mediana"]].head())


,ciudad,barrio,p25,mediana,p75,rango_intercuartil,rango_intercuartil_pct_mediana
0,Medellín,El Poblado,4400000,5700000,7700000,3300000,57.894737
1,Medellín,Poblado,4100000,5650000,8000000,3900000,69.026549
2,Medellín,Laureles,3300000,4200000,5300000,2000000,47.619048
3,Medellín,Castropol,4200000,5350000,7000000,2800000,52.336449
4,Medellín,El Tesoro,5800000,7700000,11500000,5700000,74.025974


## 7. Responder las preguntas de negocio con tablas

**Pregunta 1:** resumen por ciudad.

Se usa la mediana de las medianas barriales; no se denomina mediana global de apartamentos.

In [7]:
# Pregunta 1: resumen por ciudad.
# Se usa la mediana de las medianas barriales; no se denomina mediana global de apartamentos.
tabla_ciudad = (
    df.groupby("ciudad", dropna=False)
      .agg(
          barrios=("barrio", "nunique"),
          avisos_reportados=("avisos", "sum"),
          mediana_barrial_mediana=("mediana", "median"),
          p25_barrial_mediana=("mediana", lambda s: s.quantile(0.25)),
          p75_barrial_mediana=("mediana", lambda s: s.quantile(0.75)),
          mediana_rango_intercuartil=("rango_intercuartil", "median")
      )
      .reset_index()
      .sort_values("mediana_barrial_mediana", ascending=False)
)
display(tabla_ciudad)


,ciudad,barrios,avisos_reportados,mediana_barrial_mediana,p25_barrial_mediana,p75_barrial_mediana,mediana_rango_intercuartil
4,Cartagena,29,2721,3500000.0,2800000.0,4800000.0,2070500.0
10,Medellín,501,119582,3100000.0,2400000.0,4300000.0,1050000.0
13,Pereira,17,1654,3000000.0,2275000.0,3500000.0,1230000.0
14,Rionegro,60,11439,2600000.0,1750000.0,3075000.0,600000.0
0,Barranquilla,66,6667,2510000.0,1725000.0,3462500.0,1278250.0
1,Bogotá,439,53691,2200000.0,1600000.0,4000000.0,1037500.0
2,Bucaramanga,67,3664,1900000.0,1500000.0,2590000.0,806250.0
12,Neiva,4,98,1800000.0,1291500.0,2215000.0,1071744.0
9,Manizales,19,662,1775000.0,1500000.0,2542500.0,1125000.0
5,Chía,30,1391,1600000.0,1500000.0,1954000.0,499750.0


**Pregunta 2:** barrios ordenados por mediana del precio pedido.

In [8]:
# Pregunta 2: barrios ordenados por mediana del precio pedido.
tabla_barrio = df[[
    "ciudad", "barrio", "avisos", "p25", "mediana", "p75",
    "rango_intercuartil"
]].sort_values(["mediana", "avisos"], ascending=[False, False]).reset_index(drop=True)
display(tabla_barrio.head(20))


,ciudad,barrio,avisos,p25,mediana,p75,rango_intercuartil
0,Bogotá,Cerros de los Alpes,24,17000000,20500000,25000000,8000000
1,Bogotá,La Cabrera Chico,17,11000000,18000000,19873000,8873000
2,Bogotá,Bosques del Marquez,16,14392000,17000000,17000000,2608000
3,Bogotá,Bosques del Marques,24,14392000,15800000,18000000,3608000
4,Bogotá,Cabrera,252,10500000,15000000,18000000,7500000
5,Bogotá,Santa Barbara Alta,222,8850000,15000000,19000000,10150000
6,Bogotá,Montearroyo,31,12800000,14500000,18000000,5200000
7,Bogotá,Santa Ana Oriental,167,10050000,14000000,17000000,6950000
8,Bogotá,Rosales Bajo,36,10950000,13600000,15125000,4175000
9,Bogotá,Chico la Cabrera,57,10000000,13500000,16000000,6000000


**Pregunta 3:** dispersión central por ciudad.

In [9]:
# Pregunta 3: dispersión central por ciudad.
tabla_dispersion = (
    df.groupby("ciudad", dropna=False)
      .agg(
          barrios=("barrio", "nunique"),
          mediana_rango_intercuartil=("rango_intercuartil", "median"),
          p75_rango_intercuartil=("rango_intercuartil", lambda s: s.quantile(0.75)),
          mediana_rango_relativo_pct=("rango_intercuartil_pct_mediana", "median")
      )
      .reset_index()
      .sort_values("mediana_rango_intercuartil", ascending=False)
)
display(tabla_dispersion)


,ciudad,barrios,mediana_rango_intercuartil,p75_rango_intercuartil,mediana_rango_relativo_pct
4,Cartagena,29,2070500.0,2817500.0,54.000000
0,Barranquilla,66,1278250.0,2278000.0,51.923155
13,Pereira,17,1230000.0,1830000.0,42.376318
9,Manizales,19,1125000.0,1562500.0,56.666667
12,Neiva,4,1071744.0,1983000.0,56.429395
10,Medellín,501,1050000.0,1800000.0,34.027778
1,Bogotá,439,1037500.0,2616500.0,43.358084
2,Bucaramanga,67,806250.0,1405125.0,43.243243
14,Rionegro,60,600000.0,1162858.5,24.199199
3,Cali,170,512500.0,1307500.0,39.791667


## 8. Interpretación de resultados

In [10]:
ciudad_mayor = tabla_ciudad.iloc[0] if not tabla_ciudad.empty else None
ciudad_menor = tabla_ciudad.iloc[-1] if not tabla_ciudad.empty else None
barrio_mayor = tabla_barrio.iloc[0] if not tabla_barrio.empty else None
barrio_menor = tabla_barrio.iloc[-1] if not tabla_barrio.empty else None

if ciudad_mayor is not None:
    print(f"Mayor mediana de medianas barriales: {ciudad_mayor['ciudad']} "
          f"({ciudad_mayor['mediana_barrial_mediana']:,.0f} COP/mes).")
    print(f"Menor mediana de medianas barriales: {ciudad_menor['ciudad']} "
          f"({ciudad_menor['mediana_barrial_mediana']:,.0f} COP/mes).")
if barrio_mayor is not None:
    print(f"Barrio con mayor mediana en la tabla: {barrio_mayor['barrio']}, "
          f"{barrio_mayor['ciudad']} ({barrio_mayor['mediana']:,.0f} COP/mes).")
    print(f"Barrio con menor mediana en la tabla: {barrio_menor['barrio']}, "
          f"{barrio_menor['ciudad']} ({barrio_menor['mediana']:,.0f} COP/mes).")
print("Interpreta los valores como precios pedidos publicados. La mediana de medianas barriales "
      "resume barrios y no equivale a una mediana calculada sobre todos los apartamentos.")


Mayor mediana de medianas barriales: Cartagena (3,500,000 COP/mes).
Menor mediana de medianas barriales: Soacha (750,000 COP/mes).
Barrio con mayor mediana en la tabla: Cerros de los Alpes, Bogotá (20,500,000 COP/mes).
Barrio con menor mediana en la tabla: Frontera, Medellín (400,000 COP/mes).
Interpreta los valores como precios pedidos publicados. La mediana de medianas barriales resume barrios y no equivale a una mediana calculada sobre todos los apartamentos.


## 9. Validaciones finales y comparación antes/después

In [11]:
diagnostico_despues = {
    "filas": len(df),
    "columnas": len(df.columns),
    "nulos": int(df.isna().sum().sum()),
    "duplicados_exactos": int(df.duplicated().sum())
}
comparacion = pd.DataFrame([
    {"indicador": k, "antes": diagnostico_antes[k], "despues": diagnostico_despues[k]}
    for k in diagnostico_antes
])
display(comparacion)
assert len(df) == len(df_original), "Se modificó el número de filas sin documentar una regla."
assert set(df_original.columns).issubset(set(df.columns)), "Se perdieron columnas originales."
assert (tabla_ciudad["avisos_reportados"].sum() == df["avisos"].sum()), "Los totales de avisos no coinciden."
assert np.isclose(tabla_ciudad["barrios"].sum(), df.groupby("ciudad")["barrio"].nunique().sum())
print("Validaciones de conservación de filas, columnas fuente y totales: correctas.")


,indicador,antes,despues
0,filas,1460,1460
1,columnas,10,16
2,nulos,0,0
3,duplicados_exactos,0,0


Validaciones de conservación de filas, columnas fuente y totales: correctas.


## 10. Exportar dataset preparado y tablas CSV

In [12]:
ruta_preparado = CARPETA_DATOS / "dataset_arriendos_preparado.csv"
ruta_ciudad = CARPETA_RESULTADOS / "tabla_resultados_por_ciudad.csv"
ruta_barrio = CARPETA_RESULTADOS / "tabla_resultados_por_barrio.csv"
ruta_dispersion = CARPETA_RESULTADOS / "tabla_dispersion_por_ciudad.csv"

df.to_csv(ruta_preparado, index=False, encoding="utf-8-sig")
tabla_ciudad.to_csv(ruta_ciudad, index=False, encoding="utf-8-sig")
tabla_barrio.to_csv(ruta_barrio, index=False, encoding="utf-8-sig")
tabla_dispersion.to_csv(ruta_dispersion, index=False, encoding="utf-8-sig")

for ruta in [ruta_preparado, ruta_ciudad, ruta_barrio, ruta_dispersion]:
    print(ruta, "->", ruta.stat().st_size, "bytes")


AE3/datos/dataset_arriendos_preparado.csv -> 218457 bytes
AE3/resultados/tabla_resultados_por_ciudad.csv -> 1072 bytes
AE3/resultados/tabla_resultados_por_barrio.csv -> 83050 bytes
AE3/resultados/tabla_dispersion_por_ciudad.csv -> 916 bytes


## 11. Generar automáticamente el resumen descriptivo `.txt` del DataFrame final

In [13]:
ruta_txt = CARPETA_RESULTADOS / "resumen_dataframe_AE3.txt"
lineas = [
    "RESUMEN DEL DATAFRAME TRANSFORMADO Y ENRIQUECIDO – AE3",
    "=" * 78,
    f"Dataset: {DATASET_ID}",
    f"Configuración: {CONFIGURACION}",
    f"Split: {SPLIT}",
    f"Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "Unidad de análisis: registro agregado por ciudad y barrio.",
    "",
    f"Filas: {df.shape[0]}",
    f"Columnas: {df.shape[1]}",
    "",
    "COLUMNAS Y TIPOS"
]
lineas += [f"- {c}: {t}" for c, t in df.dtypes.items()]
lineas += ["", "VALORES NULOS POR COLUMNA"]
lineas += [f"- {c}: {int(n)}" for c, n in df.isna().sum().items()]
lineas += ["", "ESTADÍSTICAS DESCRIPTIVAS (VARIABLES NUMÉRICAS)",
           df.describe(include=[np.number]).round(2).to_string(),
           "", "VALIDACIONES",
           f"- Filas originales: {len(df_original)}",
           f"- Filas finales: {len(df)}",
           f"- Duplicados exactos finales: {int(df.duplicated().sum())}",
           f"- Registros con regla de consistencia incumplida: {int(registros_inconsistentes.sum())}",
           "", "INDICADORES DERIVADOS",
           "- rango_intercuartil = p75 - p25 (COP/mes)",
           "- rango_intercuartil_pct_mediana = (p75 - p25) / mediana * 100",
           "- participacion_avisos_parqueadero = con_parqueadero / avisos * 100",
           "- participacion_avisos_telefono = con_telefono / avisos * 100",
           "", "Fuente: https://huggingface.co/datasets/aptologia/precios-arriendo-colombia",
           "Los precios son precios pedidos publicados, no necesariamente precios finales pactados."]
ruta_txt.write_text("\n".join(lineas) + "\n", encoding="utf-8")
print("Resumen AE3 generado:", ruta_txt)
print(ruta_txt.read_text(encoding="utf-8"))


Resumen AE3 generado: AE3/resultados/resumen_dataframe_AE3.txt
RESUMEN DEL DATAFRAME TRANSFORMADO Y ENRIQUECIDO – AE3
Dataset: aptologia/precios-arriendo-colombia
Configuración: precios_arriendo_barrios
Split: train
Fecha de generación: 2026-11-05 18:17:29
Unidad de análisis: registro agregado por ciudad y barrio.

Filas: 1460
Columnas: 16

COLUMNAS Y TIPOS
- fecha: datetime64[ns]
- ciudad_id: string
- ciudad: string
- barrio: string
- avisos: int64
- p25: int64
- mediana: int64
- p75: int64
- con_parqueadero: int64
- con_telefono: int64
- ciudad_normalizada: string
- barrio_normalizado: string
- rango_intercuartil: int64
- rango_intercuartil_pct_mediana: float64
- participacion_avisos_parqueadero: float64
- participacion_avisos_telefono: float64

VALORES NULOS POR COLUMNA
- fecha: 0
- ciudad_id: 0
- ciudad: 0
- barrio: 0
- avisos: 0
- p25: 0
- mediana: 0
- p75: 0
- con_parqueadero: 0
- con_telefono: 0
- ciudad_normalizada: 0
- barrio_normalizado: 0
- rango_intercuartil: 0
- rango_inte

**Descargar el resumen TXT a tu computador (Google Colab)**

In [15]:
# Descargar el resumen TXT a tu computador (Google Colab)
from google.colab import files
files.download(str(ruta_txt))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Descargar las tablas de los resultados CSV**

tabla_resultados_por_ciudad.csv

tabla_resultados_por_barrio.csv

tabla_dispersion_por_ciudad.csv

In [16]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED

archivo_zip = CARPETA_RESULTADOS / "tablas_resultados_AE3.zip"

with ZipFile(archivo_zip, "w", ZIP_DEFLATED) as zipf:
    for ruta in [ruta_ciudad, ruta_barrio, ruta_dispersion]:
        zipf.write(ruta, arcname=ruta.name)

print("Archivo ZIP creado:", archivo_zip)
files.download(str(archivo_zip))

Archivo ZIP creado: AE3/resultados/tablas_resultados_AE3.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Diccionario actualizado de variables

| Variable | Significado | Tipo / unidad |
|---|---|---|
| `fecha` | Fecha de fotografía | fecha |
| `ciudad_id`, `ciudad`, `barrio` | Identificación geográfica | texto |
| `avisos` | Número de avisos considerados | conteo |
| `p25`, `mediana`, `p75` | Percentiles del precio pedido | COP/mes |
| `con_parqueadero`, `con_telefono` | Conteos de avisos con atributo | conteo |
| `ciudad_normalizada`, `barrio_normalizado` | Texto auxiliar normalizado | texto |
| `rango_intercuartil` | `p75 - p25` | COP/mes |
| `rango_intercuartil_pct_mediana` | Rango intercuartílico relativo a mediana | porcentaje |
| `participacion_avisos_parqueadero` | Avisos con parqueadero / avisos | porcentaje |
| `participacion_avisos_telefono` | Avisos con teléfono / avisos | porcentaje |

## 13. Limitaciones
- Los registros son agregados por barrio y ciudad, no inmuebles individuales.
- Los valores son precios pedidos de avisos; no necesariamente precios pactados.
- El resumen por ciudad utiliza la mediana de las medianas barriales; no debe interpretarse como mediana de todos los apartamentos.
- No hay una serie temporal suficiente para inferir tendencias históricas.
- Los indicadores de atributos son proporciones de conteos reportados; no permiten atribuir diferencias de precio a parqueadero o teléfono.
- Las conclusiones se limitan a la cobertura y fecha de la fotografía del dataset.

## Referencias
- Aptología. *Precios de arriendo por barrio en Colombia*. https://huggingface.co/datasets/aptologia/precios-arriendo-colombia
- Aptología. *Datos abiertos*. https://www.aptologia.com/datos-abiertos
- Hugging Face. *Load a dataset*. https://huggingface.co/docs/datasets/loading
- pandas. *User guide*. https://pandas.pydata.org/docs/
